# Train v1 · проверка базовой очистки текста

Сравниваем все 10 000 записей исходного train и очищенной версии. В таблицах ниже видны контрольные суммы, число срабатываний каждого правила, примеры изменений и все оставленные смешанные формы.

Выберите ядро проекта `.venv` и нажмите **Run All**, чтобы повторить проверку. Ноутбук читает готовые файлы. Исходный train и версия очистки лежат в `data/raw` и `data/processed/train_text_v1`.

## 1. Сверяем файлы по манифесту

Сверяем SHA-256 источника, очищенной выборки и правил сборки. Затем читаем все строки JSONL и все записи Parquet.

In [ ]:
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "data/raw/public-train-annotated.jsonl").is_file()
)
sys.path.insert(0, str(root / "dataset_review"))

from build_train_text_v1 import TEXT_FIELDS, sha256
from train_text_cleaning import PRESERVED_NAMES_AND_CODES, mixed_tokens, STEPS

manifest_path = root / "data/processed/train_text_v1/manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
source = manifest["source"]
dataset = manifest["dataset"]
attachments = {source["path"]: source["sha256"], dataset["path"]: dataset["sha256"]}
attachments.update(manifest["build_files_sha256"])
integrity = []
for relative_path, expected_sha in attachments.items():
    actual_sha = sha256(root / relative_path)
    integrity.append({
        "Файл": relative_path, "SHA-256": actual_sha, "Совпадает": expected_sha == actual_sha
    })
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(integrity))
assert all(check["Совпадает"] for check in integrity)

with (root / source["path"]).open(encoding="utf-8") as stream:
    raw = [json.loads(line) for line in stream]
clean = pq.read_table(root / dataset["path"]).to_pylist()
assert len(raw) == source["records"] == len(clean) == dataset["records"]
print(f"Источник: {len(raw):,} записей. Версия {manifest['version']}: {len(clean):,} записей.")

## 2. Проверяем происхождение записей и разметку

Для каждой строки проверяем исходный `item_id`, номер строки JSONL с единицы и весь список микрокатегорий, включая числовые ID, подписи и порядок. Пустые поля и повторные ID считаем отдельно.

In [ ]:
errors = Counter()
label_counts = {"До": Counter(), "После": Counter()}
for source_line, (original, processed) in enumerate(zip(raw, clean, strict=True), 1):
    errors["Неверный номер исходной строки"] += processed["source_line"] != source_line
    errors["Другой item_id"] += processed["item_id"] != original["item_id"]
    errors["Изменена разметка"] += processed["found_microcategories"] != original["found_microcategories"]
    for field in TEXT_FIELDS:
        errors[f"Пустое поле {field}"] += not processed[field].strip()
    for label, row in (("До", original), ("После", processed)):
        label_counts[label].update(mark["mcId"] for mark in row["found_microcategories"])

errors["Повторные item_id"] = len(clean) - len({row["item_id"] for row in clean})
display(pd.Series(errors, name="Ошибок").to_frame())
display(pd.DataFrame(label_counts).sort_index().rename_axis("mcId"))
assert not any(errors.values())

## 3. Считаем изменения и повторяем очистку

Для каждого текста прогоняем правила по порядку и сопоставляем результат с файлом. Затем применяем правила к уже очищенному тексту. После второго прохода число срабатываний каждого правила должно стать нулевым.

In [ ]:
rule_counts = {"До": Counter(), "После": Counter()}
changed_text = Counter()
mismatched = 0
for original, processed in zip(raw, clean, strict=True):
    changed_text["Объявления"] += any(original[field] != processed[field] for field in TEXT_FIELDS)
    for field in TEXT_FIELDS:
        raw_text = original[field]
        clean_text = processed[field]
        changed_text[field] += raw_text != clean_text
        for name, transform, _ in STEPS:
            transformed = transform(raw_text)
            rule_counts["До"][name, field] += transformed != raw_text
            raw_text = transformed

            transformed = transform(clean_text)
            rule_counts["После"][name, field] += transformed != clean_text
            clean_text = transformed
        mismatched += raw_text != processed[field]

display(pd.Series(changed_text, name="Изменилось").to_frame())
display(pd.DataFrame([
    {"Правило": description, "Поле": field,
     "Срабатываний до": rule_counts["До"][name, field],
     "Срабатываний после": rule_counts["После"][name, field]}
    for name, _, description in STEPS for field in TEXT_FIELDS
]))
assert mismatched == 0
assert not any(rule_counts["После"].values())
assert changed_text["Объявления"] == manifest["changed_records"]
assert all(changed_text[field] == manifest["changed_fields"][field] for field in TEXT_FIELDS)
assert all(
    rule_counts["До"][rule["id"], field] == rule["changed_fields"][field]
    for rule in manifest["transformations"] for field in TEXT_FIELDS
)

## 4. Смешанные слова и оставленные названия

Считаем слова из обеих письменностей, включая фрагменты внутри идентификаторов. После очистки остаются только перечисленные ниже названия и коды. В этой таблице можно найти каждое их вхождение по `source_line` и `item_id`.

In [ ]:
before_mixed = Counter()
after_mixed = Counter()
preserved = []
for original, processed in zip(raw, clean, strict=True):
    for field in TEXT_FIELDS:
        before_mixed.update(mixed_tokens(original[field]))
        for token in mixed_tokens(processed[field]):
            after_mixed[token] += 1
            preserved.append({
                "source_line": processed["source_line"], "item_id": processed["item_id"],
                "Поле": field, "Смешанный фрагмент": token
            })

display(pd.DataFrame({
    "До": [before_mixed.total(), len(before_mixed)],
    "После": [after_mixed.total(), len(after_mixed)]
}, index=["Вхождений", "Различных форм"]))
display(pd.DataFrame(preserved))
assert set(after_mixed) <= PRESERVED_NAMES_AND_CODES
assert dict(after_mixed) == manifest["preserved_mixed_names_and_codes"]

## 5. Примеры до и после

Здесь показаны фрагменты из реальных строк. Чтобы увидеть объявление целиком, укажите его исходный номер в `line_to_inspect` и повторно запустите последнюю ячейку раздела.

In [ ]:
samples = [
    (87, "description", "РЕMOHT"), (95, "description", "рeмонт"),
    (393, "description", "под 🔑"), (2138, "description", "WhаtsАрр"),
    (2308, "description", "5️⃣5️⃣"), (5867, "description", r"\u2028"),
    (6510, "description", "шпатлëвка"),
]

examples = []
for line_no, field, anchor in samples:
    original = raw[line_no - 1][field]
    start = original.index(anchor)
    snippet = original[max(0, start - 35):start + len(anchor) + 55]
    cleaned_snippet = snippet
    for _, transform, _ in STEPS:
        cleaned_snippet = transform(cleaned_snippet)
    examples.append({
        "source_line": line_no, "item_id": clean[line_no - 1]["item_id"],
        "До": snippet.replace("\n", " ⏎ ").replace("\r", " ⏎ "),
        "После": cleaned_snippet.replace("\n", " ⏎ ")
    })

titles = [
    {"source_line": row["source_line"], "item_id": row["item_id"],
     "До": original["title"], "После": row["title"]}
    for original, row in zip(raw, clean, strict=True) if original["title"] != row["title"]
]
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(titles))
    display(pd.DataFrame(examples))

In [ ]:
line_to_inspect = 393
assert 1 <= line_to_inspect <= len(clean)
original = raw[line_to_inspect - 1]
processed = clean[line_to_inspect - 1]
print(f"Исходная строка {line_to_inspect} · item_id {processed['item_id']}")
print("\nДО\n")
print(original["title"], original["description"], sep="\n\n")
print("\nПОСЛЕ\n")
print(processed["title"], processed["description"], sep="\n\n")

## 6. Сравниваем точные дубликаты

Очистка может объединить ранее разные тексты. Считаем группы с точным совпадением и отдельно выводим группы описаний с разными наборами `mcId`, которые возникли после объединения ранее различных описаний. Любая такая группа заслуживает отдельного разбора при формировании выборки.

In [ ]:
def grouped(rows, fields):
    groups = defaultdict(list)
    for index, row in enumerate(rows):
        groups[tuple(row[field] for field in fields)].append(index)
    return [indices for indices in groups.values() if len(indices) > 1]


def conflicting(groups):
    return [
        indices for indices in groups
        if len({tuple(sorted(mark["mcId"] for mark in raw[index]["found_microcategories"]))
                for index in indices}) > 1
    ]


comparisons = []
new_description_conflicts = []
for fields in (("description",), TEXT_FIELDS):
    original_groups = grouped(raw, fields)
    clean_groups = grouped(clean, fields)
    conflicts = conflicting(clean_groups)
    merged_conflicts = [
        indices for indices in conflicts
        if len({tuple(raw[index][field] for field in fields) for index in indices}) > 1
    ]
    comparisons.append({
        "Ключ": " + ".join(fields),
        "Дубликаты до": len(original_groups), "Дубликаты после": len(clean_groups),
        "Конфликты до": len(conflicting(original_groups)), "Конфликты после": len(conflicts),
        "Конфликты с объединением разных исходных текстов": len(merged_conflicts)
    })
    if fields == ("description",):
        new_description_conflicts = merged_conflicts

display(pd.DataFrame(comparisons))
if new_description_conflicts:
    display(pd.DataFrame([
        {"Группа": group_number, "source_line": index + 1, "item_id": raw[index]["item_id"],
         "mcId": [mark["mcId"] for mark in raw[index]["found_microcategories"]]}
        for group_number, indices in enumerate(new_description_conflicts, 1)
        for index in indices
    ]))
print("Проверки целостности, разметки, происхождения строк и повторной очистки пройдены.")

Проверки показывают эффект очистки текста. Влияние на качество модели можно оценить при сравнении двух обучений с одинаковым разбиением. SEO, складские приписки и дубликаты остаются материалом для отдельных версий.